In [6]:
import pandas as pd
import shutil
import json
from pathlib import Path

# Configuration
dataset_name = "energy_transition_example"
base_path = Path("./outputs")
output_dir = Path("./CSV_output") / dataset_name
esm_capacity_path = output_dir / "capacity_addition_aggregated_by_location.csv"
system_path = base_path.parent / dataset_name / "system.json"

original_dir = Path(r"C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_HP\ZEN-Model_HP")
new_dir = Path(r"C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_HP\ZEN-Model_HP_new")
original_dyv_path = original_dir / "set_carriers" / "HP" / "demand_yearly_variation.csv"
target_dyv_path = new_dir / "set_carriers" / "HP" / "demand_yearly_variation.csv"

technology = "heat_pump"
unit_multiplier = 1_000_000  # Convert GW to kW
country_map = {
    "DE": "DEU",
    "AT": "AUT",
    "IT": "ITA",
    "CZ": "CZE",
    "ROE": "ROE",
}

# Clone Input Directory
if not new_dir.exists():
    shutil.copytree(original_dir, new_dir)
    print(f"Cloned input to: {new_dir}")
else:
    print(f"Directory already exists: {new_dir}")

# Load System Metadata
with open(system_path, "r") as f:
    config = json.load(f)

year_labels = [str(config["reference_year"] + i * config["interval_between_years"])
               for i in range(config["optimized_years"])]

# Load Data
df_existing = pd.read_csv(original_dyv_path)
df_esm = pd.read_csv(esm_capacity_path)
df_filtered = df_esm[df_esm["technology"] == technology]

# Group and Map
df_grouped = df_filtered.groupby("location").sum(numeric_only=True).reset_index()
df_grouped["node"] = df_grouped["location"].map(country_map)
df_grouped = df_grouped[df_grouped["node"].notna()].drop(columns="location")

# Transpose and Merge
df_new = df_grouped.set_index("node").T
df_new.index.name = "year"
df_new = df_new.reset_index()
df_new = df_new[df_new["year"].isin(year_labels)]
df_new["year"] = df_new["year"].astype(str)
df_existing["year"] = df_existing["year"].astype(str)

df_new = df_new.set_index("year") * unit_multiplier
df_existing = df_existing.set_index("year")

for col in df_new.columns.intersection(df_existing.columns):
    df_existing[col] = df_new[col].combine_first(df_existing[col])

# Save final file
df_result = df_existing.reset_index()
df_result.iloc[:, 1:] = df_result.iloc[:, 1:].round(3)
df_result.to_csv(target_dyv_path, index=False)
print(f"demand_yearly_variation.csv written to:\n{target_dyv_path}")


Directory already exists: C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_HP\ZEN-Model_HP_new
demand_yearly_variation.csv written to:
C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_HP\ZEN-Model_HP_new\set_carriers\HP\demand_yearly_variation.csv
